# RL Optimization

This notebook now orchestrates the reusable RL pipeline code in `src/rl_pipeline.py`.


In [ ]:
from pathlib import Path

import pandas as pd
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv

from src.rl_pipeline import (
    DQNConfig,
    InventoryEnvConfig,
    build_dqn_agent,
    evaluate_policy,
    make_category_env,
    make_env,
    run_episode,
    sS_policy,
    eoq_policy,
)

ROOT = Path.cwd()


## 1. Load Processed Artifacts


In [ ]:
hourly_df = pd.read_parquet(ROOT / 'artifacts' / 'hourly_features.parquet')
forecast_path = ROOT / 'artifacts' / 'rl_forecast_features.parquet'
forecast_df = pd.read_parquet(forecast_path) if forecast_path.exists() else None
hourly_df.head()


## 2. Environment Sanity Checks


In [ ]:
env_config = InventoryEnvConfig()
sample_uid = hourly_df['unique_id'].iloc[0]
sample_env = make_env(sample_uid, hourly_df, split='train', forecast_df=forecast_df, config=env_config)
obs, _ = sample_env.reset()
print('InventoryEnv obs shape:', obs.shape)

sample_category = int(hourly_df['first_category_id'].mode().iloc[0])
category_env = make_category_env(sample_category, hourly_df, split='train', forecast_df=forecast_df, config=env_config, min_series_days=5)
obs, _ = category_env.reset()
print('CategoryInventoryEnv obs shape:', obs.shape)


## 3. Train One Per-Category Agent


In [ ]:
dqn_config = DQNConfig()
train_env = DummyVecEnv([lambda: make_category_env(sample_category, hourly_df, split='train', forecast_df=forecast_df, config=env_config, min_series_days=5)])
val_env = DummyVecEnv([lambda: make_category_env(sample_category, hourly_df, split='val', forecast_df=forecast_df, config=env_config, min_series_days=5)])

agent = build_dqn_agent(train_env, dqn_config)
eval_cb = EvalCallback(val_env, eval_freq=2000, n_eval_episodes=3, deterministic=True, verbose=0)
# agent.learn(total_timesteps=20_000, callback=eval_cb, progress_bar=True)
agent


## 4. Compare Against Baselines


In [ ]:
test_env = make_category_env(sample_category, hourly_df, split='test', forecast_df=forecast_df, config=env_config, min_series_days=5)
results = {
    '(s,S)': evaluate_policy(test_env, lambda obs, env: sS_policy(obs, env, s=10, S=30), n_episodes=3),
    'EOQ': evaluate_policy(test_env, lambda obs, env: eoq_policy(obs, env, Q=15, r=8), n_episodes=3),
}
results


## 5. Inspect A Single Rollout


In [ ]:
history = run_episode(test_env, lambda obs, env: sS_policy(obs, env, s=10, S=30))
pd.DataFrame(history).head()
